# Compare Dafne (Water) to Ground Truth

In [1]:
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', 'dafne_water_segs')
GT_BASE    = os.path.join('..', 'myosegmenTUM')
RESULT_DIR = 'results_on_water'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, gt_label_idx, dafne_npz_key)
MUSCLES = [
    ('R_gracilis',  5, 'Gracilis_R'),
    ('L_gracilis',  1, 'Gracilis_L'),
    ('R_sartorius', 8, 'Sartorius_R'),
    ('L_sartorius', 4, 'Sartorius_L'),
]

npz_files = sorted(f for f in os.listdir(SEG_DIR) if f.endswith('_dafne_thigh.npz'))
print(f'Seg dir   : {os.path.abspath(SEG_DIR)}')
print(f'Result dir: {os.path.abspath(RESULT_DIR)}')
print(f'Found     : {len(npz_files)} npz files')

Seg dir   : C:\Projects\dissector\eval_notebooks\dafne_water_segs
Result dir: C:\Projects\dissector\eval_notebooks\dafne_thigh_results\results_on_water
Found     : 46 npz files


## Evaluate all muscles

In [3]:
def evaluate_muscle(muscle_name, gt_label_idx, dafne_key):
    results = []
    for npz_file in npz_files:
        stem      = npz_file.replace('_dafne_thigh.npz', '')  # HV001_1_WATER_stack1
        subject   = re.split(r'_WATER_', stem)[0]             # HV001_1
        m         = re.search(r'stack(\d+)', stem)
        if not m:
            print(f'  could not parse stack number: {npz_file}, skipping')
            continue
        stack_num = m.group(1)

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        npz_data = np.load(os.path.join(SEG_DIR, npz_file))
        pred_arr = npz_data[dafne_key].astype(float)

        pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {npz_file}: empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'image':                                gt_path,
            'pred_label':                           npz_file,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    csv_path = os.path.join(RESULT_DIR, f'df_{muscle_name}_dafne_water.csv')
    df.to_csv(csv_path)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

dfs = {}
for muscle_name, gt_idx, dafne_key in MUSCLES:
    print(f'\n── {muscle_name} ──')
    dfs[muscle_name] = evaluate_muscle(muscle_name, gt_idx, dafne_key)

print('\nDone.')


── R_gracilis ──
  Saved 46 rows → results_on_water\df_R_gracilis_dafne_water.csv

── L_gracilis ──
  Saved 46 rows → results_on_water\df_L_gracilis_dafne_water.csv

── R_sartorius ──
  Saved 46 rows → results_on_water\df_R_sartorius_dafne_water.csv

── L_sartorius ──
  Saved 46 rows → results_on_water\df_L_sartorius_dafne_water.csv

Done.


## Results

In [ ]:
for name, df in dfs.items():
    print(f'\n── {name} ──')
    display(df[['pred_label', f'{name}_dice', f'{name}_hausdorff']].head())